In [ ]:
# 1.1 Библиотеки и устройство
import os, sys, re, threading, gzip, json, math, time, copy, random, glob, shutil, tarfile, hashlib, contextlib
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score, precision_recall_curve
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    major, _ = torch.cuda.get_device_capability()
    # bfloat16 есть у Ampere и новее (A100, L4). У T4 есть только float16, а в широких слоях этой сети он переполняется
    # (NaN/inf, обучение расходится; то же писал автор бейзлайна), поэтому на T4 считаем в обычном float32: медленнее, но надёжно
    AMP_DTYPE = torch.bfloat16 if major >= 8 else None
    torch.backends.cudnn.benchmark = True                       # подбирает самые быстрые свёрточные алгоритмы под наши размеры
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True   # TF32 на A100/L4: быстрее float32, точности хватает
    print("GPU:", torch.cuda.get_device_name(), "| точность:", AMP_DTYPE or "float32 (смешанная точность выключена: нет bfloat16)")
else:
    AMP_DTYPE = None
    print("GPU нет: обучение на CPU пригодно только для проверки кода (FAST_DEV)")

def autocast():
    # контекст смешанной точности; на CPU ничего не делает
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE is not None else contextlib.nullcontext()

In [ ]:
# 1.2 Все настраиваемые параметры в одном месте
SPECIES  = os.environ.get("EPIC_SPECIES", "Ofas_2.0")        # "jaNemVect1.1" (учебный, есть ответы теста) или "Ofas_2.0" (наш вид)
FAST_DEV = os.environ.get("EPIC_FAST_DEV") == "1"                # крошечные настройки: только проверить, что весь код проходит
SESSION_T0 = time.time()                                         # начало сессии: от него считается STOP_AFTER_HOURS

# --- где мы запущены и куда складывать файлы ---
ON_KAGGLE = os.path.isdir("/kaggle/working")
ON_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = ON_COLAB          # в Colab чекпоинты пишем на Google Диск: они переживут отключение сессии
if USE_GOOGLE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
if ON_KAGGLE:   default_data, default_out = "/tmp/epic", f"/kaggle/working/epic_runs/{SPECIES}"
elif USE_GOOGLE_DRIVE: default_data, default_out = "/content/epic", f"/content/drive/MyDrive/epic_runs/{SPECIES}"
else:           default_data, default_out = "/content/epic", f"/content/epic_runs/{SPECIES}"
DATA_DIR = os.environ.get("EPIC_DATA_DIR", default_data)         # куда скачивать и распаковывать архив (большой, не храним)
OUT_DIR  = os.environ.get("EPIC_OUT_DIR", default_out)           # чекпоинты, предсказания, сабмит
BASE = f"{DATA_DIR}/{SPECIES}"
os.makedirs(OUT_DIR, exist_ok=True)

# --- чекпоинты, принесённые с другой платформы (например, из Colab в Kaggle через Kaggle Dataset) ---
IMPORT_CKPT_GLOB = os.environ.get("EPIC_IMPORT_CKPT", "/kaggle/input/**/*.pt" if ON_KAGGLE else "")
for src in sorted(glob.glob(IMPORT_CKPT_GLOB, recursive=True)) if IMPORT_CKPT_GLOB else []:
    dst = f"{OUT_DIR}/{os.path.basename(src)}"
    if os.path.basename(src).startswith(SPECIES + "_") and not os.path.exists(dst):
        shutil.copy(src, dst); print("импортирован чекпоинт:", src)

CFG = dict(
    # --- данные ---
    WIN=8192,                 # длина окна ДНК на входе модели, нт
    CROP=512,                 # края окна (с каждой стороны), где не считаем loss и не берём предсказания: там мало контекста
    HOLDOUT_FRAC=0.05,        # доля train-whitelist в каждой из двух отложенных групп контигов (tune и report)
    USE_LOWERCASE=True,       # подавать ли модели флаг «строчная буква» (мягкая маска повторов)
    POS_FRAC=0.5,             # доля окон в батче, построенных вокруг положительной позиции
    # --- модель ---
    CH_LOCAL=96,              # каналы ветки базового разрешения
    TOWER=(128, 192, 256, 320),   # каналы уровней башни; каждый уровень уменьшает разрешение в STRIDE раз
    STRIDE=4,
    ATTN_LAYERS=2,            # слоёв self-attention на самом грубом уровне (0 = как в бейзлайне)
    ATTN_HEADS=8,
    REPLICATE_HEADS=True,     # дополнительные выходы на каждую реплику
    AUX_WEIGHT=0.25,          # вес loss-а реплик относительно основного (сумма реплик)
    # --- обучение ---
    STEPS=48000 if SPECIES == "jaNemVect1.1" else 96000,   # шагов на одну модель (у бейзлайна на Ofas_2.0 было 96000)
    BATCH=8,
    LR=2e-3, WEIGHT_DECAY=1e-2, WARMUP=0.05,
    ATTN_LR_MULT=0.25,        # attention учится с LR в 4 раза меньше: трансформерные слои неустойчивы при LR свёрток
    DIVERGE_FRAC=0.5,         # если AUPRC (EMA) на tune упал ниже этой доли от лучшего -> откат к лучшим весам и LR пополам
    EMA_DECAY=0.999,          # 0 = без EMA
    EVAL_EVERY=4000,          # как часто проверять качество на группе tune
    EVAL_BP=2_000_000,        # сколько нт группы tune брать в быструю проверку во время обучения
    N_SEEDS=2,                # моделей в ансамбле
    ALL_DATA_SEEDS=(),        # сиды, которые учатся на ВСЕХ train-контигах (включая tune/report); в оценку на них не идут
    CKPT_EVERY=1000,          # каждые столько шагов сохраняем полное состояние обучения (можно продолжить с этого места)
    STOP_AFTER_HOURS=11.5 if ON_KAGGLE else None,   # аккуратно остановиться (с сохранением) через столько часов от начала сессии
    # --- предсказание и постобработка ---
    TTA_SHIFTS=2,             # сколько сдвинутых нарезок окон усреднять
    PRED_BATCH=32,            # окон за один проход при предсказании (на каждое окно ещё и обратная цепь)
    ALPHAS=(0.0, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8),   # сетка показателя rank-power
    KEEP_FRAC=0.995,          # alpha = наибольшее, при котором AUPRC на tune >= KEEP_FRAC * лучшего
    EXTRA_ALPHAS=(),          # дополнительно записать сабмиты с этими alpha (крупнее alpha: выше Spearman, чуть ниже AUPRC)
)
if FAST_DEV:
    CFG.update(WIN=1024, CROP=128, CH_LOCAL=16, TOWER=(16, 24, 32, 48), ATTN_HEADS=4, STEPS=60, BATCH=4,
               EVAL_EVERY=30, EVAL_BP=60_000, PRED_BATCH=8, CKPT_EVERY=20)

# прогон «ансамбль из 4»: seed 0 и 1 берутся готовыми из датасета epic-v2-ofas-ckpt, 2 и 3 учатся на всех контигах
CFG.update(N_SEEDS=4, ALL_DATA_SEEDS=(2, 3), EXTRA_ALPHAS=(0.70, 0.75))

CFG_HELP = {
    "WIN": "длина входного окна, нт", "CROP": "отбрасываемые края окна, нт", "HOLDOUT_FRAC": "размер каждой отложенной группы",
    "USE_LOWERCASE": "флаг строчной буквы на входе", "POS_FRAC": "доля окон вокруг положительных позиций",
    "CH_LOCAL": "каналы ветки 1 нт", "TOWER": "каналы уровней башни", "STRIDE": "сжатие на уровень",
    "ATTN_LAYERS": "слоёв attention", "ATTN_HEADS": "голов attention", "REPLICATE_HEADS": "выходы на реплики",
    "AUX_WEIGHT": "вес loss реплик", "STEPS": "шагов на модель", "BATCH": "окон в батче", "LR": "макс. скорость обучения",
    "WEIGHT_DECAY": "L2-регуляризация AdamW", "WARMUP": "доля шагов разогрева", "EMA_DECAY": "затухание EMA",
    "ATTN_LR_MULT": "множитель LR для attention", "DIVERGE_FRAC": "порог отката при расхождении",
    "EVAL_EVERY": "период проверки", "EVAL_BP": "нт в быстрой проверке", "N_SEEDS": "моделей в ансамбле",
    "ALL_DATA_SEEDS": "сиды, обучаемые на всех контигах", "EXTRA_ALPHAS": "доп. сабмиты с другими alpha",
    "CKPT_EVERY": "период сохранения состояния", "STOP_AFTER_HOURS": "плановая остановка, часов",
    "TTA_SHIFTS": "сдвигов нарезки при предсказании", "PRED_BATCH": "окон в батче предсказания",
    "ALPHAS": "сетка alpha", "KEEP_FRAC": "допустимая потеря AUPRC при квантовании",
}
print("платформа:", "Kaggle" if ON_KAGGLE else "Colab" if ON_COLAB else "другая", "| вид:", SPECIES, "| FAST_DEV:", FAST_DEV)
print("данные:", BASE, "\nрезультаты и чекпоинты:", OUT_DIR)
display(pd.DataFrame({"значение": {k: str(v) for k, v in CFG.items()}, "что это": CFG_HELP}))

In [ ]:
# 1.3 Скачивание и распаковка архива вида с Zenodo (doi:10.5281/zenodo.22285753), с проверкой MD5
def download_dataset(species, data_dir):
    if os.path.isdir(f"{data_dir}/{species}"):
        print("датасет уже распакован:", f"{data_dir}/{species}"); return
    import requests
    os.makedirs(data_dir, exist_ok=True)
    rec = requests.get("https://zenodo.org/api/records/22285753", timeout=60).json()
    f = [x for x in rec["files"] if x["key"] == f"{species}.tgz"][0]
    dst = f"{data_dir}/{species}.tgz"
    md5 = hashlib.md5()
    with requests.get(f["links"]["self"], stream=True, timeout=120) as r, open(dst, "wb") as fh:
        r.raise_for_status()
        for chunk in tqdm(r.iter_content(1 << 22), total=f["size"] // (1 << 22) + 1, desc=f"скачиваю {species}", unit="×4МБ"):
            fh.write(chunk); md5.update(chunk)
    want = f.get("checksum", "md5:").split(":")[-1]
    assert not want or md5.hexdigest() == want, "MD5 не совпал: архив повреждён, удалите его и скачайте снова"
    with tarfile.open(dst) as tf:
        try: tf.extractall(data_dir, filter="data")
        except TypeError: tf.extractall(data_dir)
    os.remove(dst)   # архив больше не нужен, экономим диск

download_dataset(SPECIES, DATA_DIR)
for p in sorted(glob.glob(f"{BASE}/**/*", recursive=True)):
    if os.path.isfile(p): print(f"{os.path.getsize(p) / 1e6:9.1f} МБ  {os.path.relpath(p, BASE)}")

In [ ]:
# 2.1 Интервалы whitelist, шаблон теста и геном
FP = f"{BASE}/for_participants"

def read_bed(path, ncols):
    names = ["chrom", "start", "end", "name", "score", "strand"][:ncols]
    return pd.read_csv(path, sep="\t", header=None, usecols=range(ncols), names=names, dtype={"chrom": str, "strand": str})

wl_train = read_bed(f"{FP}/whitelist.train.bed.gz", 3)
tpl_test = read_bed(f"{FP}/template.test.bed.gz", 6)
TRAIN_CONTIGS = list(dict.fromkeys(wl_train.chrom))
TEST_CONTIGS = list(dict.fromkeys(tpl_test.chrom))

LUT = np.full(256, 8, np.uint8)            # любой символ кроме ACGT/acgt -> 8 (N)
for i, b in enumerate("ACGT"):
    LUT[ord(b)] = i; LUT[ord(b.lower())] = i + 4

def read_genome(path, keep):
    # FASTA -> {контиг: uint8-массив кодов}; читаем только нужные контиги
    genome, name, chunks = {}, None, []
    def flush():
        if name in keep: genome[name] = LUT[np.frombuffer(b"".join(chunks), np.uint8)]
    with gzip.open(path, "rb") as fh:
        for line in tqdm(fh, desc="читаю геном", unit=" строк", mininterval=5):
            if line.startswith(b">"):
                flush(); name = line[1:].split()[0].decode(); chunks = []
            elif name in keep:
                chunks.append(line.rstrip())
    flush()
    return genome

genome = read_genome(f"{BASE}/genome/genome.fa.gz", set(TRAIN_CONTIGS) | set(TEST_CONTIGS))
assert all(c in genome for c in TRAIN_CONTIGS + TEST_CONTIGS), "в fasta нет части контигов из whitelist"

IN_WL = {c: np.zeros(len(genome[c]), bool) for c in TRAIN_CONTIGS}   # True = позиция в whitelist.train
for c, s, e in zip(wl_train.chrom, wl_train.start, wl_train.end):
    IN_WL[c][s:e] = True

TEST_LEN = int((tpl_test.end - tpl_test.start).sum())
print(f"train: {len(TRAIN_CONTIGS)} контигов, whitelist {int((wl_train.end - wl_train.start).sum()) / 1e6:.1f} Мб")
print(f"test : {len(TEST_CONTIGS)} контигов, строк в шаблоне {len(tpl_test)}, позиций в сабмите {TEST_LEN:,}")

In [ ]:
# 2.2 Метки: разворачиваем RLE-отрезки BED и объединяем реплики
# имена бывают с тканью: csRNA-r1-train.bed.gz (Nematostella) или csRNA-Embryo-r1-train.bed.gz (Ofas_2.0)
REP_FILES = sorted(glob.glob(f"{FP}/csRNA-*r[0-9]-train.bed.gz"))
assert REP_FILES, f"не найдены файлы меток csRNA-*rN-train.bed.gz в {FP}"
N_REP = len(REP_FILES)
REP_COLS = [f"r{i + 1}" for i in range(N_REP)]

def read_counts_bed(path):
    # строка BED = отрезок [start, end) с одинаковым счётом -> по строке на каждую позицию отрезка
    d = pd.read_csv(path, sep="\t", header=None, usecols=[0, 1, 2, 4, 5], names=["chrom", "start", "end", "count", "strand"],
                    dtype={"chrom": str, "strand": str})
    span = (d.end - d.start).values
    first = np.repeat(np.cumsum(span) - span, span)               # индекс начала своего отрезка для каждой развёрнутой позиции
    return pd.DataFrame({"chrom": np.repeat(d.chrom.values, span),
                         "pos": np.repeat(d.start.values, span) + (np.arange(span.sum()) - first),
                         "strand": np.repeat((d.strand.values == "-").astype(np.int8), span),
                         "count": np.repeat(d["count"].values, span).astype(np.float32)}), len(d)

lab = None
for col, path in zip(REP_COLS, REP_FILES):
    r, n_lines = read_counts_bed(path)
    print(f"{os.path.basename(path)}: строк {n_lines:,} -> позиций {len(r):,} (+{len(r) / n_lines - 1:.1%} после разворота отрезков)")
    r = r.rename(columns={"count": col})
    lab = r if lab is None else lab.merge(r, on=["chrom", "pos", "strand"], how="outer")
lab[REP_COLS] = lab[REP_COLS].fillna(0).astype(np.float32)
lab["tot"] = lab[REP_COLS].sum(axis=1)

TRACK = {}   # контиг -> отсортированные позиции, цепь (0 = +, 1 = −), сумма реплик и каждая реплика
for c, d in lab.groupby("chrom", sort=False):
    d = d.sort_values(["pos", "strand"])
    TRACK[c] = dict(pos=d.pos.values.astype(np.int64), strand=d.strand.values.astype(np.int64),
                    tot=d.tot.values.astype(np.float32), reps=d[REP_COLS].values.T.astype(np.float32))

outside = sum(int((~IN_WL[c][t["pos"]]).sum()) for c, t in TRACK.items() if c in IN_WL)
print(f"положительных позиций-цепей: {len(lab):,}; вне whitelist: {outside}; реплик: {N_REP}")
if N_REP < 2 and CFG["REPLICATE_HEADS"]:
    CFG["REPLICATE_HEADS"] = False; print("реплика одна -> REPLICATE_HEADS выключен")

In [ ]:
# 3.1 Жадный выбор групп tune и report
wl_bp = (wl_train.assign(bp=wl_train.end - wl_train.start).groupby("chrom", sort=False).bp.sum())
pos_n = pd.Series({c: len(TRACK[c]["pos"]) if c in TRACK else 0 for c in wl_bp.index})
GLOBAL_RATE = pos_n.sum() / (2 * wl_bp.sum())

def pick_group(avail, target_bp):
    sel, s_bp, s_pos = [], 0, 0
    while s_bp < target_bp and avail:
        best = min(avail, key=lambda c: abs((s_pos + pos_n[c]) / (2 * (s_bp + wl_bp[c])) - GLOBAL_RATE))
        sel.append(best); s_bp += wl_bp[best]; s_pos += pos_n[best]; avail.remove(best)
    return sel

min_bp = max(min(100_000, int(wl_bp.max())), int(wl_bp.sum()) // 1000)   # совсем мелкие контиги в hold-out не берём
avail = [c for c in wl_bp.index if wl_bp[c] >= min_bp]
GROUPS = {g: pick_group(avail, CFG["HOLDOUT_FRAC"] * wl_bp.sum()) for g in ("tune", "report")}
FIT_CONTIGS = [c for c in TRAIN_CONTIGS if c not in set(GROUPS["tune"]) | set(GROUPS["report"])]

summary = pd.DataFrame({g: {"контигов": len(cs), "Мб whitelist": wl_bp[cs].sum() / 1e6, "доля положительных": pos_n[cs].sum() / (2 * wl_bp[cs].sum())}
                        for g, cs in [("обучение", FIT_CONTIGS), ("tune", GROUPS["tune"]), ("report", GROUPS["report"])]}).T
display(summary.style.format({"Мб whitelist": "{:.1f}", "доля положительных": "{:.2e}"}))
print("tune  :", GROUPS["tune"][:10], "..." if len(GROUPS["tune"]) > 10 else "")
print("report:", GROUPS["report"][:10], "..." if len(GROUPS["report"]) > 10 else "")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].pie(summary["Мб whitelist"], labels=summary.index, autopct="%.0f%%", colors=["tab:blue", "tab:orange", "tab:green"])
ax[0].set_title("Как поделён train-whitelist")
ax[1].bar(summary.index, summary["доля положительных"], color=["tab:blue", "tab:orange", "tab:green"])
ax[1].axhline(GLOBAL_RATE, ls="--", color="k", lw=1, label="среднее по виду"); ax[1].legend()
ax[1].set_title("Доля положительных позиций-цепей в группах")
plt.tight_layout(); plt.show()

In [ ]:
# 7.1 Строки шаблона, предсказание по контигам, сбор в порядке шаблона
def make_rows(df):
    # df: chrom, start, end, strand ('+'/'-' или 0/1) -> добавляем длину и смещение строки в итоговом векторе
    r = df[["chrom", "start", "end"]].copy()
    st = df["strand"]
    r["strand"] = st.values.astype(int) if pd.api.types.is_integer_dtype(st) else (st.astype(str).values == "-").astype(int)
    r["len"] = r.end - r.start; r["off"] = np.r_[0, np.cumsum(r.len.values)[:-1]]
    return r.reset_index(drop=True)

def train_rows(contigs):
    iv = wl_train[wl_train.chrom.isin(set(contigs))]
    return make_rows(pd.concat([iv.assign(strand=0), iv.assign(strand=1)]))

def gather_rows(rows, contig_fn, dtype=np.float32, desc=None):
    # contig_fn(c, rows_c, lo, hi) -> массив (2, hi-lo); раскладываем его в порядок строк
    out = np.zeros(int(rows.len.sum()), dtype)
    groups = rows.groupby("chrom", sort=False)
    for c, g in tqdm(groups, total=groups.ngroups, desc=desc, disable=desc is None, mininterval=2):
        lo, hi = int(g.start.min()), int(g.end.max()); arr = contig_fn(c, g, lo, hi)
        for s, e, st, o in zip(g.start.values, g.end.values, g.strand.values, g.off.values):
            out[o:o + e - s] = arr[st, s - lo:e - lo]
    return out

def rows_nmask(rows):
    return gather_rows(rows, lambda c, g, lo, hi: np.broadcast_to(genome[c][lo:hi] == 8, (2, hi - lo)), bool)

def rows_targets(rows):
    def fn(c, g, lo, hi):
        a = np.zeros((2, hi - lo), np.float32); t = TRACK.get(c)
        if t is not None:
            k = (t["pos"] >= lo) & (t["pos"] < hi); a[t["strand"][k], t["pos"][k] - lo] = t["tot"][k]
        return a
    return gather_rows(rows, fn)

@torch.no_grad()
def predict_contig(models, c, g, lo, hi, cfg):
    W, C = cfg["WIN"], cfg["CROP"]; step = W - 2 * C
    iv = np.unique(np.stack([g.start.values, g.end.values], 1), axis=0)          # строки по возрастанию start
    ms, me = iv[:, 0], np.maximum.accumulate(iv[:, 1])
    def overlaps(a, b):                                          # задевает ли [a, b) хоть один интервал
        k = np.searchsorted(me, a, side="right"); return k < len(ms) and ms[k] < b
    starts = [s for k in range(cfg["TTA_SHIFTS"]) for s in range(lo - C - k * step // cfg["TTA_SHIFTS"], hi - C, step) if overlaps(s + C, s + C + step)]
    acc = np.zeros((2, hi - lo), np.float32); cnt = np.zeros(hi - lo, np.float32)
    for i in range(0, len(starts), cfg["PRED_BATCH"]):
        chunk = starts[i:i + cfg["PRED_BATCH"]]
        lr = predict_windows(models, np.stack([window_codes(c, s, W) for s in chunk]), cfg)[:, :, C:W - C]
        for s, p in zip(chunk, lr):
            a, b = max(s + C, lo), min(s + C + step, hi)
            if b > a: acc[:, a - lo:b - lo] += p[:, a - s - C:b - s - C]; cnt[a - lo:b - lo] += 1
    return acc / np.maximum(cnt, 1)

def predict_rows(models, rows, cfg=CFG, desc="предсказание"):
    return gather_rows(rows, lambda c, g, lo, hi: predict_contig(models, c, g, lo, hi, cfg), desc=desc)

In [ ]:
# 7.2 Предсказание групп tune и report: каждая модель отдельно и ансамбль; подбор alpha на tune; честная оценка на report
# модели, обученные на всех контигах (ALL_DATA_SEEDS), видели tune и report: в подбор alpha и честную оценку они не идут
HO = {}
for gname in ("report",):
    rows = train_rows(GROUPS[gname])
    HO[gname] = dict(rows=rows, gt=rows_targets(rows), n=rows_nmask(rows))
    fn = glob.glob("/kaggle/input/**/report_preds_v2moth.npy", recursive=True)[0]
    P = np.load(fn).astype(np.float32); print("предсказания:", fn, P.shape, "строк:", int(rows.len.sum()))
    assert P.shape[-1] == int(rows.len.sum()), "длина предсказаний не совпала с hold-out report"
    HO[gname]["ens"] = P.mean(0) if P.ndim == 2 else P


In [ ]:
"""Официальные метрики EPIC и запись сабмита (та же логика, что в epic_v2_model.ipynb, раздел 6.1 и 8.3).

* Сабмит: по одному числу в [0, 1] с 5 знаками после запятой на каждую позицию шаблона, поэтому скорер видит не больше
  100 001 различных уровней.
* AUPRC = average precision по всем позициям (равные значения идут одной группой).
* Spearman = корреляция Пирсона между *плотными* рангами (dense rank: равные значения получают один ранг, ранги идут
  подряд без пропусков) предсказания и правды, только в позициях, где правда > 0.
"""
import gzip
import numpy as np

SCALE = 100_000


def quantise(pred01):
    return np.rint(np.clip(pred01, 0.0, 1.0) * SCALE).astype(np.int32)


def prauc_from_levels(levels, labels):
    pos = np.bincount(levels[labels], minlength=SCALE + 1)[::-1].astype(np.int64)
    tot = np.bincount(levels, minlength=SCALE + 1)[::-1].astype(np.int64)
    keep = tot > 0; tp, n = np.cumsum(pos)[keep], np.cumsum(tot)[keep]
    P = tp[-1] if len(tp) else 0
    if P == 0: return 0.0
    return float((np.diff(np.r_[0, tp]) / P * (tp / n)).sum())


def dense_rank(x):
    return np.unique(x, return_inverse=True)[1].astype(np.float64)


def spearman_dense(pred_pos, gt_pos):
    if len(pred_pos) < 2: return 0.0
    a, b = dense_rank(pred_pos), dense_rank(gt_pos)
    a -= a.mean(); b -= b.mean(); d = np.sqrt((a * a).sum() * (b * b).sum())
    return 0.0 if d == 0 else float(a @ b / d)


def official_score(pred01, gt, mask=None):
    """Метрики так, как их считает скорер по файлу сабмита (pred01 — значения в [0, 1])."""
    if mask is not None: pred01, gt = pred01[mask], gt[mask]
    lv, y = quantise(pred01), gt > 0
    return {"AUPRC": prauc_from_levels(lv, y), "Spearman": spearman_dense(lv[y], gt[y])}


def rank_power(score, alpha, nmask=None):
    """Монотонно переводим любой счёт в [0, 1]: позиция с местом r в рейтинге (1 = лучшая) получает r^(-alpha).
    Порядок позиций не меняется, но при округлении до 5 знаков все позиции ниже некоторого места сливаются в одно
    значение (и в один плотный ранг). alpha задаёт, сколько верхних мест сохраняют различимость. N-позиции -> 0."""
    x = score.astype(np.float32, copy=True)
    if nmask is not None: x[nmask] = -np.inf
    order = np.argsort(-x, kind="stable")
    ranks = np.empty(len(x), np.int32); ranks[order] = np.arange(1, len(x) + 1, dtype=np.int32); del order
    s = (1.0 - (ranks - 1) / len(x)) if alpha == 0 else np.power(ranks.astype(np.float32), -alpha)
    s = np.rint(np.clip(s, 0, 1) * SCALE).astype(np.float32) / SCALE
    if nmask is not None: s[nmask] = 0.0
    return s


def write_submission(values01, path, chunk=8_000_000):
    """Каждая строка ровно 'd.ddddd\\n', как '%.5f', но быстро."""
    ints = np.rint(np.clip(values01, 0, 1) * SCALE).astype(np.int64)
    with gzip.open(path, "wb", compresslevel=4) as fh:
        for a in range(0, len(ints), chunk):
            v = ints[a:a + chunk]; buf = np.empty((len(v), 8), np.uint8)
            buf[:, 0] = ord("0") + v // SCALE; buf[:, 1] = ord("."); buf[:, 7] = ord("\n")
            frac = v % SCALE
            for k in range(6, 1, -1): buf[:, k] = ord("0") + frac % 10; frac //= 10
            fh.write(buf.tobytes())

# Дешёвые проверки «что даёт знание об ошибках» на готовых предсказаниях (CPU, без обучения сети).
# run(z, gt, ok, fold, base, prev, out): z — стандартизованный лог-скор, fold — 0/1/-1 (на какой половине позиция),
# base/prev — коды нуклеотида в позиции и предыдущего в направлении цепи (0..3 ACGT, 4+ иначе).
# Схема: параметры и alpha выбираются на одной половине, качество измеряется на другой (обе направления), метрики официальные.
import json, itertools, numpy as np
from scipy.ndimage import gaussian_filter1d, maximum_filter1d, uniform_filter1d
ALPHAS = [0.6, 0.75, 0.9]

def run(z, gt, ok, fold, base, prev, out, cand_frac=0.03):
    z = z.astype(np.float32); rows = []; idx = {f: np.flatnonzero((fold == f) & ok) for f in (0, 1)}
    def measure(name, params, s):
        for a in ALPHAS:
            for f in (0, 1):
                r = rank_power(s[idx[f]], a); m = official_score(r, gt[idx[f]])
                rows.append(dict(variant=name, params=params, alpha=a, fold=f, **m))
    measure("baseline", "", z)
    for sg in (1, 2, 3): measure("gauss", f"sigma={sg}", gaussian_filter1d(z, sg).astype(np.float32))
    for k, d in itertools.product((2, 5), (0.25, 0.5, 1.0)):
        mx = maximum_filter1d(z, 2 * k + 1); measure("nms", f"k={k},delta={d}", z - d * (z < mx).astype(np.float32))
    for k, w in itertools.product((2, 5, 10), (0.25, 0.5, 1.0)):
        measure("neighbor_max", f"k={k},w={w}", z + w * maximum_filter1d(z, 2 * k + 1))
    pyr = ((prev == 1) | (prev == 3)) & (base == 0)           # пара CA/TA в позиции старта
    for c in (0.25, 0.5, 1.0): measure("ca_penalty", f"c={c}", z - c * pyr.astype(np.float32))
    # --- reranker (LightGBM) на кандидатах: верхние cand_frac по z
    import lightgbm as lgb
    thr = np.quantile(z[ok], 1 - cand_frac); cand = ok & (z >= thr)
    ind = (z >= thr).astype(np.float32)
    def feats(use_seq, use_ctx):
        F = [z]
        if use_ctx:
            for k in (1, 2, 5, 10, 50): F += [maximum_filter1d(z, 2 * k + 1) - z]
            F += [uniform_filter1d(z, 101) - z, uniform_filter1d(ind, 101), uniform_filter1d(ind, 21)]
        if use_seq: F += [base.astype(np.float32), prev.astype(np.float32), pyr.astype(np.float32)]
        return np.stack(F, 1)
    for name, us, uc in (("rerank_ctx", False, True), ("rerank_seq", True, False), ("rerank_ctx_seq", True, True)):
        X_all = feats(us, uc); P = {}
        for tr in (0, 1):
            itr = np.flatnonzero(cand & (fold == tr)); m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, verbose=-1)
            m.fit(X_all[itr], (gt[itr] > 0).astype(int)); P[tr] = m
        for tr in (0, 1):
            ev = 1 - tr; s = z.copy(); ic = np.flatnonzero(cand & (fold == ev))
            s[ic] = z.max() + 1 + P[tr].predict_proba(X_all[ic])[:, 1]
            for a in ALPHAS:
                r = rank_power(s[idx[ev]], a); rows.append(dict(variant=name, params=f"trained_on_fold{tr}", alpha=a, fold=ev, **official_score(r, gt[idx[ev]])))
    import pandas as pd
    df = pd.DataFrame(rows); df.to_csv(out + ".grid.csv", index=False)
    # выбор параметров и alpha на «tune»-половине, оценка на другой
    res = []
    for v, g in df.groupby("variant"):
        for ev in (0, 1):
            tr = 1 - ev
            if v.startswith("rerank"):
                best = g[(g.fold == ev)].sort_values("AUPRC", ascending=False)  # reranker уже обучен на другой половине; alpha выбираем на tr-половине baseline-подобно: берём alpha лучшего baseline
                a = df[(df.variant == "baseline") & (df.fold == tr)].sort_values("AUPRC", ascending=False).iloc[0].alpha
                pick = g[(g.fold == ev) & (g.alpha == a)].iloc[0]
            else:
                gt_ = g[g.fold == tr].sort_values("AUPRC", ascending=False).iloc[0]
                pick = g[(g.fold == ev) & (g.params == gt_.params) & (g.alpha == gt_.alpha)].iloc[0]
            res.append(dict(variant=v, params=pick.params, alpha=pick.alpha, eval_fold=ev, AUPRC=pick.AUPRC, Spearman=pick.Spearman))
    R = pd.DataFrame(res); R.to_csv(out + ".summary.csv", index=False)
    S = R.groupby("variant")[["AUPRC", "Spearman"]].mean(); b = S.loc["baseline"]
    S["dAUPRC_%"] = (S.AUPRC / b.AUPRC - 1) * 100; S["dSpearman"] = S.Spearman - b.Spearman
    S.to_csv(out + ".table.csv"); print(S.round(4).to_string()); print(R.to_string())
    return S

def seq_codes(chrom, start, end, strand, genome, total):
    """Коды основания в позиции и предыдущего (в направлении цепи) для всех позиций, склеенных по строкам. strand: 0 (+) / 1 (-)."""
    COMP = np.array([3, 2, 1, 0, 4, 5, 6, 7, 8]); base = np.empty(total, np.uint8); prev = np.empty(total, np.uint8); o = 0
    for c, s, e, st in zip(chrom, start, end, strand):
        g = genome[c]; ext = np.pad(g[max(s - 1, 0):e + 1], (1 if s == 0 else 0, max(e + 1 - len(g), 0)), constant_values=8); n = e - s
        b, p_ = (COMP[ext[1:n + 1]], COMP[ext[2:n + 2]]) if st else (ext[1:n + 1], ext[0:n])
        base[o:o + n] = b % 4 if False else np.where(b < 8, b % 4, 4); prev[o:o + n] = np.where(p_ < 8, p_ % 4, 4); o += n
    return base, prev

def local_norm(z, beta, R, seg=None):
    """P-1: score - beta * (локальное среднее score в ±R). seg=(off, len) — среднее считается внутри каждой строки (контиг/цепь), не через границы."""
    if seg is None: return (z - beta * uniform_filter1d(z, 2 * R + 1, mode="nearest")).astype(np.float32)
    out = np.empty_like(z, dtype=np.float32)
    for o, l in zip(*seg): out[o:o + l] = z[o:o + l] - beta * uniform_filter1d(z[o:o + l], 2 * R + 1, mode="nearest")
    return out

def run_p1(z, gt, ok, fold, out, seg=None, Rs=(2000, 5000, 10000, 20000), betas=(0.25, 0.5, 0.75, 1.0), alphas=ALPHAS, max_sp_loss=0.01):
    """P-1: делительная/вычитающая локальная нормировка. Параметры (R, beta, alpha) выбираются на одной половине (макс. AUPRC при потере
    Spearman <= max_sp_loss против лучшего baseline этой половины), оценка на другой; обе стороны. Критерий приёмки: AUPRC +3% при потере Spearman < 0.01."""
    import pandas as pd
    z = z.astype(np.float32); idx = {f: np.flatnonzero((fold == f) & ok) for f in (0, 1)}; rows = []
    def measure(R, beta, s):
        for a in alphas:
            for f in (0, 1):
                m = official_score(rank_power(s[idx[f]], a), gt[idx[f]]); rows.append(dict(R=R, beta=beta, alpha=a, fold=f, **m))
    measure(0, 0.0, z)
    for R, b in itertools.product(Rs, betas): measure(R, b, local_norm(z, b, R, seg)); print("R", R, "beta", b, flush=True)
    df = pd.DataFrame(rows); df.to_csv(out + ".grid.csv", index=False); res = []
    for ev in (0, 1):
        tr = 1 - ev; g = df[df.fold == tr]; bt = g[g.R == 0].sort_values("AUPRC", ascending=False).iloc[0]
        cand = g[(g.R > 0) & (g.Spearman >= bt.Spearman - max_sp_loss)]
        pick = (cand if len(cand) else g[g.R > 0]).sort_values("AUPRC", ascending=False).iloc[0]
        be = df[(df.fold == ev) & (df.R == 0) & (df.alpha == bt.alpha)].iloc[0]
        pe = df[(df.fold == ev) & (df.R == pick.R) & (df.beta == pick.beta) & (df.alpha == pick.alpha)].iloc[0]
        res.append(dict(eval_fold=ev, R=pick.R, beta=pick.beta, alpha=pick.alpha, base_alpha=bt.alpha, AUPRC_base=be.AUPRC, AUPRC=pe.AUPRC,
                        Spearman_base=be.Spearman, Spearman=pe.Spearman, dAUPRC_pct=(pe.AUPRC / be.AUPRC - 1) * 100, dSpearman=pe.Spearman - be.Spearman))
    R_ = pd.DataFrame(res); R_.to_csv(out + ".summary.csv", index=False); print(R_.round(4).to_string())
    print("СРЕДНЕЕ dAUPRC_pct = %.2f, dSpearman = %.4f" % (R_.dAUPRC_pct.mean(), R_.dSpearman.mean()))
    return R_


rep = HO["report"]; rows = rep["rows"]; ok = ~rep["n"]; p = rep["ens"].astype(np.float64)
z = ((p - p[ok].mean()) / p[ok].std()).astype(np.float32)
ch = rows.chrom.values; uq = list(dict.fromkeys(ch)); cf = {c: i % 2 for i, c in enumerate(uq)}
fold = np.empty(len(p), np.int8)
for c, o_, l_ in zip(ch, rows.off.values, rows.len.values): fold[o_:o_ + l_] = cf[c]
base, prev = seq_codes(ch, rows.start.values, rows.end.values, rows.strand.values, genome, len(p))
seg = (rows.off.values, rows.len.values)
run_p1(z, rep["gt"], ok, fold, "/kaggle/working/p1_v2", seg=seg)
